# Importing Libraries

In [12]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# Importing the Dataset

In [13]:
train_data = pd.read_csv("/content/Corona_NLP_train.csv", encoding='latin-1')
test_data = pd.read_csv("/content/Corona_NLP_test.csv", encoding='latin-1')

In [14]:
train_data.head()

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
3,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive
4,3803,48755,NaN,16-03-2020,"Me, ready to go at supermarket during the #COV...",Extremely Negative


In [15]:
test_data.head()

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,1,44953,NYC,02-03-2020,TRENDING: New Yorkers encounter empty supermar...,Extremely Negative
1,2,44954,"Seattle, WA",02-03-2020,When I couldn't find hand sanitizer at Fred Me...,Positive
2,3,44955,NaN,02-03-2020,Find out how you can protect yourself and love...,Extremely Positive
3,4,44956,Chicagoland,02-03-2020,#Panic buying hits #NewYork City as anxious sh...,Negative
4,5,44957,"Melbourne, Victoria",03-03-2020,#toiletpaper #dunnypaper #coronavirus #coronav...,Neutral


# Preprocessing Dataset

In [16]:
train_data = train_data.drop(["UserName", "ScreenName", "Location", "TweetAt"], axis=1)
train_data.head(5)

,OriginalTweet,Sentiment
0,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,advice Talk to your neighbours family to excha...,Positive
2,Coronavirus Australia: Woolworths to give elde...,Positive
3,My food stock is not the only one which is emp...,Positive
4,"Me, ready to go at supermarket during the #COV...",Extremely Negative


In [17]:
test_data = test_data.drop(["UserName", "ScreenName", "Location", "TweetAt"], axis=1)
test_data.head(5)

,OriginalTweet,Sentiment
0,TRENDING: New Yorkers encounter empty supermar...,Extremely Negative
1,When I couldn't find hand sanitizer at Fred Me...,Positive
2,Find out how you can protect yourself and love...,Extremely Positive
3,#Panic buying hits #NewYork City as anxious sh...,Negative
4,#toiletpaper #dunnypaper #coronavirus #coronav...,Neutral


# Data Encoding

In [18]:
le = preprocessing.LabelEncoder()
le.fit(train_data['Sentiment'])
train_data['Sentiment'] = le.transform(train_data['Sentiment'])
test_data['Sentiment'] = le.transform(test_data['Sentiment'])

In [19]:
train_data.head(5)

,OriginalTweet,Sentiment
0,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,3
1,advice Talk to your neighbours family to excha...,4
2,Coronavirus Australia: Woolworths to give elde...,4
3,My food stock is not the only one which is emp...,4
4,"Me, ready to go at supermarket during the #COV...",0


In [20]:
test_data.head(5)

,OriginalTweet,Sentiment
0,TRENDING: New Yorkers encounter empty supermar...,0
1,When I couldn't find hand sanitizer at Fred Me...,4
2,Find out how you can protect yourself and love...,1
3,#Panic buying hits #NewYork City as anxious sh...,2
4,#toiletpaper #dunnypaper #coronavirus #coronav...,3


In [21]:
X_train = train_data['OriginalTweet']
y_train = train_data['Sentiment']
X_test = test_data['OriginalTweet']
y_test = test_data['Sentiment']

In [22]:
y_train = to_categorical(y_train, num_classes=5)
y_test = to_categorical(y_test, num_classes=5)

In [23]:
len(X_train), len(X_test)

(41157, 3798)

# Compute text Statistics for Vectorization

In [24]:
averageWordsLength = round(sum([len(i.split()) for i in train_data['OriginalTweet']]) / len(train_data['OriginalTweet']))
totalWordsLength = len(set(" ".join(train_data['OriginalTweet']).split()))
print(f"Data Loaded. Training samples: {len(X_train)}")
print(f"Average words per message: {averageWordsLength}")
print(f"Approximate vocabulary size: {totalWordsLength}")

Data Loaded. Training samples: 41157
Average words per message: 31
Approximate vocabulary size: 136386


# TextVectorization layer for Tokenization

In [25]:
text_vectorizer = layers.TextVectorization(
    max_tokens=20000,#totalWordsLength,
    standardize="lower_and_strip_punctuation",
    output_mode="int",
    output_sequence_length=50#averageWordsLength
)
text_vectorizer.adapt(X_train)

# Embedding layer

In [26]:
embedding_layer = layers.Embedding(
    input_dim=20000,#totalWordsLength,
    output_dim=64,
    embeddings_initializer='uniform',
)

# Model Architecture

In [27]:
input_layer = layers.Input(shape=(1,), dtype=tf.string)
vectorized_layer = text_vectorizer(input_layer)
embedding_layer = embedding_layer(vectorized_layer)
x = layers.LSTM(64, dropout=0.3,
    recurrent_dropout=0.3)(embedding_layer)
# x = layers.LSTM(64)(x)
# x = layers.GlobalMaxPooling1D()(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)
output_layer = layers.Dense(5, activation='softmax')(x)

model = keras.Model(inputs=input_layer, outputs=output_layer)
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 50)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 50, 64)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,317,509 (5.03 MB)

 Trainable params: 1,317,509 (5.03 MB)

 Non-trainable params: 0 (0.00 B)

# Training & Evaluating the Model

In [30]:
history = model.fit(X_train.to_numpy(), y_train, epochs=20, batch_size=64, validation_data=(X_test.to_numpy(), y_test))

Epoch 1/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 54s 76ms/step - accuracy: 0.3076 - loss: 1.5162 - val_accuracy: 0.4107 - val_loss: 1.3509
Epoch 2/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 83s 78ms/step - accuracy: 0.4632 - loss: 1.2594 - val_accuracy: 0.5579 - val_loss: 1.0944
Epoch 3/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 80s 74ms/step - accuracy: 0.6440 - loss: 0.9321 - val_accuracy: 0.7085 - val_loss: 0.8076
Epoch 4/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 48s 75ms/step - accuracy: 0.7582 - loss: 0.6799 - val_accuracy: 0.7380 - val_loss: 0.7613
Epoch 5/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 82s 75ms/step - accuracy: 0.8078 - loss: 0.5754 - val_accuracy: 0.7430 - val_loss: 0.7640
Epoch 6/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 81s 74ms/step - accuracy: 0.8372 - loss: 0.5041 - val_accuracy: 0.7425 - val_loss: 0.7737
Epoch 7/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 48s 75ms/step - accuracy: 0.8534 - loss: 0.4595 - val_accuracy: 0.7430 - val_loss: 0.7964
Epoch 8/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 49s 75ms/step - accuracy: 0.8693 - loss: 0.4163 - 

# Sample Prediction